# Offline Language Translation System (LTS)

### Project Track: Natural Language Processing (NLP) / LLM-based System

## 1. Problem Definition & Objective

**Problem Statement:**
Most modern language translation systems rely on continuous internet connectivity and cloud-based APIs. This limits usability in low-connectivity environments and raises privacy concerns.

**Objective:**
- Build a fully offline languageC-based language translation system
- Support multiple Indian and international languages
- Provide CLI and GUI interfaces
- Ensure privacy-preserving, standalone deployment

**Real-world Relevance:**
- Rural and low-connectivity areas
- Privacy-sensitive translations
- Educational and local language accessibility

## 2. Data Understanding & Preparation

**Dataset Source:**
- Public multilingual datasets used during pretraining of IndicTrans2 models
- No custom dataset collection required (model-based translation)

**Data Nature:**
- User-provided text input
- Real-time inference

**Preprocessing Steps:**
- Language normalization
- Tokenization using pretrained tokenizer
- Indic text normalization using IndicProcessor

**Missing Values / Noise Handling:**
- Not applicable (direct user input text)

## 3. Model / System Design

**AI Technique Used:**
- NLP with Transformer-based Large Language Models (LLMs)

**Models Used:**
- Indic → English: indictrans2-indic-en-dist-200M
- English → Indic: indictrans2-en-indic-dist-200M

**Architecture:**
Input Text → Preprocessing → Tokenization → Offline Model Inference → Postprocessing → Output Text

**Design Justification:**
- Distilled 200M models for reduced memory usage
- Offline inference for privacy and availability
- English pivoting for Indic-to-Indic translation

## 4. Core Implementation

In [1]:
# Import required libraries
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

/home/snehal-modgil/anaconda3/envs/offline-lts/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 4.1 Offline Translation Engine

In [2]:
# Constants for offline models
# 200M models are faster and require less RAM for offline use
INDIC_EN_MODEL = "ai4bharat/indictrans2-indic-en-dist-200M"
EN_INDIC_MODEL = "ai4bharat/indictrans2-en-indic-dist-200M"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class OfflineTranslator:
    def __init__(self):
        print(f"Loading models on {DEVICE}...")
        
        # Initialize Processor
        self.ip = IndicProcessor(inference=True)
        
        # Load Indic -> English Model
        self.tokenizer_indic_en = AutoTokenizer.from_pretrained(INDIC_EN_MODEL, trust_remote_code=True)
        self.model_indic_en = AutoModelForSeq2SeqLM.from_pretrained(INDIC_EN_MODEL, trust_remote_code=True).to(DEVICE)
        
        # Load English -> Indic Model
        self.tokenizer_en_indic = AutoTokenizer.from_pretrained(EN_INDIC_MODEL, trust_remote_code=True)
        self.model_en_indic = AutoModelForSeq2SeqLM.from_pretrained(EN_INDIC_MODEL, trust_remote_code=True).to(DEVICE)

        self.model_indic_en.eval()
        self.model_en_indic.eval()
        
        self.lang_codes = {
            "english": "eng_Latn",
            "hindi": "hin_Deva",
            "tamil": "tam_Taml",
            "telugu": "tel_Telu",
            "bengali": "ben_Beng",
            "marathi": "mar_Deva",
            "gujarati": "guj_Gujr",
            "punjabi": "pan_Guru",
            "kannada": "kan_Knda",
            "malayalam": "mal_Mlym",
            "odia": "ory_Orya",
            "assamese": "asm_Beng",
            "urdu": "urd_Arab",
            "nepali": "npi_Deva",
            "sanskrit": "san_Deva",
            "maithili": "mai_Deva"
        }

        self.supported_langs = set(self.lang_codes.values())  

    def get_supported_languages(self):
        """
        Returns a dict: {Language Name: Language Code}
        """
        return self.lang_codes

    def _run_inference(self, text, src, tgt, model, tokenizer):
        # 1. Preprocess: This formats the string to include the correct tags
        # and avoids the "Invalid source language tag" error
        batch = self.ip.preprocess_batch([text], src_lang=src, tgt_lang=tgt)
        
        # 2. Tokenize
        inputs = tokenizer(
            batch, 
            return_tensors="pt", 
            padding=True, 
            truncation=True
        ).to(DEVICE)
        
        # 3. Generate
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs, 
                max_length=256, 
                num_beams=5, 
                use_cache=False
            )
        
        # 4. Postprocess: Decodes and cleans the output
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        return self.ip.postprocess_batch(decoded, lang=tgt)[0]
    
    def _normalize_lang(self, lang):
        # If already a code, return as-is
        if lang in self.supported_langs:
            return lang

        # If it's a name, convert to code
        if lang in self.lang_codes:
            return self.lang_codes[lang]

        raise ValueError(f"Unsupported language: {lang}")


    def translate(self, text, src_lang, tgt_lang):
        # Validate language codes (case-sensitive)
        src_lang = self._normalize_lang(src_lang)
        tgt_lang = self._normalize_lang(tgt_lang)

        if src_lang not in self.supported_langs:
            raise ValueError(f"Unsupported source language: {src_lang}")
    
        if tgt_lang not in self.supported_langs:
            raise ValueError(f"Unsupported target language: {tgt_lang}")
    
        # Case 1: Indic -> English
        if tgt_lang == "eng_Latn" and src_lang != "eng_Latn":
            return self._run_inference(
                text,
                src_lang,
                "eng_Latn",
                self.model_indic_en,
                self.tokenizer_indic_en
            )
    
        # Case 2: English -> Indic
        if src_lang == "eng_Latn" and tgt_lang != "eng_Latn":
            return self._run_inference(
                text,
                "eng_Latn",
                tgt_lang,
                self.model_en_indic,
                self.tokenizer_en_indic
            )
    
        # Case 3: Indic -> Indic (pivot through English)
        if src_lang != "eng_Latn" and tgt_lang != "eng_Latn":
            # Step A: Indic -> English
            intermediate_en = self._run_inference(
                text,
                src_lang,
                "eng_Latn",
                self.model_indic_en,
                self.tokenizer_indic_en
            )
    
            # Step B: English -> Indic
            return self._run_inference(
                intermediate_en,
                "eng_Latn",
                tgt_lang,
                self.model_en_indic,
                self.tokenizer_en_indic
            )
    
        # Case 4: Same source and target (should not happen via CLI)
        return text


if __name__ == "__main__":
    translator = OfflineTranslator()
    
    # Test 1: Hindi to English
    # print("Hindi -> English:", translator.translate("आप कैसे हैं?", "hindi", "english"))
    
    # Test 2: Hindi to Tamil (Pivoting automatically)
    # print("Hindi -> Tamil:", translator.translate("नमस्ते, आप कैसे हैं?", "hindi", "tamil"))

Loading models on cuda...


### 4.2 Command Line Interface (CLI)

In [ ]:
LANGUAGES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Bengali": "ben_Beng",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Punjabi": "pan_Guru",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Odia": "ory_Orya",
    "Assamese": "asm_Beng",
    "Urdu": "urd_Arab",
    "Nepali": "npi_Deva",
    "Sanskrit": "san_Deva",
    "Maithili": "mai_Deva"
}

def show_languages():
    print("\nSupported Languages:")
    for idx, lang in enumerate(LANGUAGES.keys(), start=1):
        print(f"{idx}. {lang}")

def get_language(prompt):
    while True:
        show_languages()
        choice = input(prompt).strip()

        # Numeric selection
        if choice.isdigit():
            idx = int(choice) - 1
            if 0 <= idx < len(LANGUAGES):
                lang_name = list(LANGUAGES.keys())[idx]
                return lang_name, LANGUAGES[lang_name]

        # Text selection
        for name in LANGUAGES:
            if choice.lower() == name.lower():
                return name, LANGUAGES[name]

        print("❌ Invalid choice. Try again.")


def main():
    print("\n=== Offline Indian Language Translation System ===")

    translator = OfflineTranslator()

    while True:
        src_name, src_code = get_language("\nSelect SOURCE language: ")
        tgt_name, tgt_code = get_language("Select TARGET language: ")

        if src_code == tgt_code:
            print("❌ Source and target languages cannot be the same.")
            continue

        text = input(f"\nEnter text in {src_name} (or type 'exit'): ").strip()
        if text.lower() == "exit":
            print("Exiting translator.")
            break

        try:
            output = translator.translate(text, src_code, tgt_code)
            print(f"\nTranslated ({tgt_name}):")
            print(output)
        except Exception as e:
            print("❌ Translation failed:", e)

if __name__ == "__main__":
    main()


=== Offline Indian Language Translation System ===
Loading models on cuda...

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili
❌ Invalid choice. Try again.

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili
❌ Invalid choice. Try again.

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili
❌ Invalid choice. Try again.

Supported Languages:
1. English
2. Hindi
3. Tamil
4. Telugu
5. Bengali
6. Marathi
7. Gujarati
8. Punjabi
9. Kannada
10. Malayalam
11. Odia
12. Assamese
13. Urdu
14. Nepali
15. Sanskrit
16. Maithili
❌ Invalid choice. Try again.

Supported

### 4.3 Graphical User Interface (GUI)

In [3]:
"""Offline Indian Language Translator - GUI

A modern, responsive Tkinter front-end for the OfflineTranslator engine.

Key design choices:
- Model loading and translation both run on background threads so the
  window never freezes, with a status bar reflecting current state.
- The output font is chosen dynamically per target language's script,
  picking the best available installed font (falls back gracefully if
  a script-specific font isn't installed, instead of silently mangling
  text into escaped \\uXXXX sequences).
- Layout uses grid with row/column weights throughout, so the window
  is genuinely resizable rather than fixed-pixel.
"""

import threading
import tkinter as tk
from tkinter import ttk, messagebox
import tkinter.font as tkfont

# Display name -> language code (kept in the same order as the CLI/README)
LANGUAGES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Bengali": "ben_Beng",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr",
    "Punjabi": "pan_Guru",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Odia": "ory_Orya",
    "Assamese": "asm_Beng",
    "Urdu": "urd_Arab",
    "Nepali": "npi_Deva",
    "Sanskrit": "san_Deva",
    "Maithili": "mai_Deva",
}

# Preferred font candidates per script (first installed match wins).
# Falls back to UI_FALLBACK_FONT if none of a script's candidates exist,
# so we never silently break rendering the way a hardcoded Windows-only
# font (e.g. "Segoe UI") did previously.
SCRIPT_FONT_CANDIDATES = {
    "Latn": ["Segoe UI", "Noto Sans", "DejaVu Sans", "Arial"],
    "Deva": ["Noto Sans Devanagari", "Lohit Devanagari", "FreeSans", "Noto Sans"],
    "Taml": ["Noto Sans Tamil", "Lohit Tamil", "Noto Sans"],
    "Telu": ["Noto Sans Telugu", "Lohit Telugu", "Noto Sans"],
    "Beng": ["Noto Sans Bengali", "Lohit Bengali", "Noto Sans"],
    "Gujr": ["Noto Sans Gujarati", "Lohit Gujarati", "Noto Sans"],
    "Guru": ["Noto Sans Gurmukhi", "Lohit Punjabi", "Noto Sans"],
    "Knda": ["Noto Sans Kannada", "Lohit Kannada", "Noto Sans"],
    "Mlym": ["Noto Sans Malayalam", "Lohit Malayalam", "Noto Sans"],
    "Orya": ["Noto Sans Oriya", "Lohit Oriya", "Noto Sans"],
    "Arab": ["Noto Sans Arabic", "Noto Naskh Arabic", "Noto Sans"],
}
UI_FALLBACK_FONT = "DejaVu Sans"

BG = "#f2f4f7"
PANEL_BG = "#ffffff"
ACCENT_GREEN = "#2e9e4f"
ACCENT_GREEN_HOVER = "#268043"
ACCENT_BLUE = "#2f7fe0"
ACCENT_BLUE_HOVER = "#2668bd"
TEXT_DARK = "#1f2430"
BORDER = "#d7dbe2"


def script_of(lang_code: str) -> str:
    """'hin_Deva' -> 'Deva'. Falls back to 'Latn' for anything unrecognised."""
    if "_" in lang_code:
        return lang_code.split("_", 1)[1]
    return "Latn"


class FontResolver:
    """Picks the best installed font for a script, caching lookups."""

    def __init__(self):
        self._installed = set(tkfont.families())
        self._cache = {}

    def for_script(self, script: str) -> str:
        if script in self._cache:
            return self._cache[script]
        for candidate in SCRIPT_FONT_CANDIDATES.get(script, []):
            if candidate in self._installed:
                self._cache[script] = candidate
                return candidate
        self._cache[script] = UI_FALLBACK_FONT
        return UI_FALLBACK_FONT

    def for_lang_code(self, lang_code: str) -> str:
        return self.for_script(script_of(lang_code))


class TranslatorGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Offline Indian Language Translator")
        self.root.geometry("1040x600")
        self.root.minsize(760, 460)
        self.root.configure(bg=BG)

        self.fonts = FontResolver()
        self.translator = None
        self.model_ready = False

        self._build_style()
        self._build_layout()
        self._set_status("Loading models...", busy=True)
        self._load_model_async()

    # ------------------------------------------------------------------
    # Styling
    # ------------------------------------------------------------------
    def _build_style(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass  # theme not available on this platform; use default

        ui_font = self.fonts.for_script("Latn")

        style.configure("TFrame", background=BG)
        style.configure("Panel.TFrame", background=PANEL_BG)
        style.configure(
            "Header.TLabel",
            background=BG,
            foreground=TEXT_DARK,
            font=(ui_font, 19, "bold"),
        )
        style.configure(
            "SectionTitle.TLabel",
            background=PANEL_BG,
            foreground=TEXT_DARK,
            font=(ui_font, 11, "bold"),
        )
        style.configure(
            "Status.TLabel",
            background=BG,
            foreground="#5b6472",
            font=(ui_font, 9),
        )
        style.configure("TCombobox", font=(ui_font, 10))

        style.configure(
            "Accent.TButton",
            background=ACCENT_GREEN,
            foreground="white",
            font=(ui_font, 10, "bold"),
            padding=(14, 8),
            borderwidth=0,
        )
        style.map(
            "Accent.TButton",
            background=[("active", ACCENT_GREEN_HOVER), ("disabled", "#9fcaab")],
        )

        style.configure(
            "Secondary.TButton",
            background=ACCENT_BLUE,
            foreground="white",
            font=(ui_font, 10, "bold"),
            padding=(14, 8),
            borderwidth=0,
        )
        style.map(
            "Secondary.TButton",
            background=[("active", ACCENT_BLUE_HOVER), ("disabled", "#a7c3ea")],
        )

        style.configure(
            "Ghost.TButton",
            background=PANEL_BG,
            foreground=TEXT_DARK,
            font=(ui_font, 10),
            padding=(10, 8),
            borderwidth=1,
        )

    # ------------------------------------------------------------------
    # Layout
    # ------------------------------------------------------------------
    def _build_layout(self):
        self.root.rowconfigure(1, weight=1)
        self.root.columnconfigure(0, weight=1)

        # ---- Header ----
        header = ttk.Frame(self.root, style="TFrame")
        header.grid(row=0, column=0, sticky="ew", padx=20, pady=(18, 8))
        header.columnconfigure(0, weight=1)

        ttk.Label(
            header,
            text="Offline Indian Language Translator",
            style="Header.TLabel",
        ).grid(row=0, column=0, sticky="w")

        self.status_var = tk.StringVar(value="Starting...")
        ttk.Label(header, textvariable=self.status_var, style="Status.TLabel").grid(
            row=1, column=0, sticky="w", pady=(2, 0)
        )

        # ---- Body: two panels ----
        body = ttk.Frame(self.root, style="TFrame")
        body.grid(row=1, column=0, sticky="nsew", padx=20, pady=(0, 8))
        body.rowconfigure(0, weight=1)
        body.columnconfigure(0, weight=1)
        body.columnconfigure(1, weight=0)  # swap button column, fixed width
        body.columnconfigure(2, weight=1)

        self.source_panel, self.source_lang_var, self.input_text = self._build_panel(
            body, "Source Language", column=0, default_lang="English"
        )
        self._build_swap_button(body, column=1)
        self.target_panel, self.target_lang_var, self.output_text = self._build_panel(
            body, "Target Language", column=2, default_lang="Hindi", editable=False
        )

        # React to target-language changes by swapping the output font
        self.target_lang_var.trace_add("write", self._on_target_lang_changed)

        # ---- Footer: actions ----
        footer = ttk.Frame(self.root, style="TFrame")
        footer.grid(row=2, column=0, sticky="ew", padx=20, pady=(0, 18))
        footer.columnconfigure(0, weight=1)
        footer.columnconfigure(1, weight=0)
        footer.columnconfigure(2, weight=0)
        footer.columnconfigure(3, weight=0)
        footer.columnconfigure(4, weight=1)

        self.clear_btn = ttk.Button(
            footer, text="Clear", style="Ghost.TButton", command=self._on_clear
        )
        self.clear_btn.grid(row=0, column=1, padx=6)

        self.translate_btn = ttk.Button(
            footer,
            text="Translate",
            style="Accent.TButton",
            command=self._on_translate,
            state="disabled",
        )
        self.translate_btn.grid(row=0, column=2, padx=6)

        self.copy_btn = ttk.Button(
            footer,
            text="Copy Output",
            style="Secondary.TButton",
            command=self._on_copy_output,
        )
        self.copy_btn.grid(row=0, column=3, padx=6)

        # Enter-to-translate convenience binding on the input box
        self.input_text.bind("<Control-Return>", lambda _e: self._on_translate())

    def _build_panel(self, parent, title, column, default_lang, editable=True):
        panel = ttk.Frame(parent, style="Panel.TFrame")
        panel.grid(row=0, column=column, sticky="nsew", padx=6, pady=4)
        panel.rowconfigure(2, weight=1)
        panel.columnconfigure(0, weight=1)

        ttk.Label(panel, text=title, style="SectionTitle.TLabel").grid(
            row=0, column=0, sticky="w", padx=12, pady=(12, 4)
        )

        lang_var = tk.StringVar(value=default_lang)
        combo = ttk.Combobox(
            panel,
            textvariable=lang_var,
            values=list(LANGUAGES.keys()),
            state="readonly",
        )
        combo.grid(row=1, column=0, sticky="ew", padx=12, pady=(0, 10))

        text_frame = ttk.Frame(panel, style="Panel.TFrame")
        text_frame.grid(row=2, column=0, sticky="nsew", padx=12, pady=(0, 12))
        text_frame.rowconfigure(0, weight=1)
        text_frame.columnconfigure(0, weight=1)

        text_widget = tk.Text(
            text_frame,
            wrap="word",
            font=(self.fonts.for_script("Latn"), 12),
            relief="solid",
            borderwidth=1,
            highlightthickness=0,
            padx=8,
            pady=8,
        )
        text_widget.grid(row=0, column=0, sticky="nsew")

        scrollbar = ttk.Scrollbar(text_frame, orient="vertical", command=text_widget.yview)
        scrollbar.grid(row=0, column=1, sticky="ns")
        text_widget.configure(yscrollcommand=scrollbar.set)

        if not editable:
            text_widget.configure(state="disabled")

        return panel, lang_var, text_widget

    def _build_swap_button(self, parent, column):
        frame = ttk.Frame(parent, style="TFrame")
        frame.grid(row=0, column=column, sticky="ns")
        frame.rowconfigure(0, weight=1)
        ttk.Button(
            frame,
            text="\u21c4",  # <-> swap glyph, ASCII-safe fallback if font lacks it
            style="Ghost.TButton",
            width=3,
            command=self._on_swap_languages,
        ).grid(row=0, column=0, pady=(70, 0))

    # ------------------------------------------------------------------
    # Model loading
    # ------------------------------------------------------------------
    def _load_model_async(self):
        def worker():
            try:
                translator = OfflineTranslator()
                error = None
            except Exception as exc:  # surfaced to the user, not swallowed
                translator = None
                error = exc
            self.root.after(0, self._on_model_loaded, translator, error)

        threading.Thread(target=worker, daemon=True).start()

    def _on_model_loaded(self, translator, error):
        if error is not None:
            self._set_status("Failed to load models.", busy=False)
            messagebox.showerror("Model Load Error", str(error))
            return
        self.translator = translator
        self.model_ready = True
        self.translate_btn.configure(state="normal")
        self._set_status("Ready.", busy=False)

    # ------------------------------------------------------------------
    # Actions
    # ------------------------------------------------------------------
    def _on_translate(self):
        if not self.model_ready or self.translator is None:
            return

        text = self.input_text.get("1.0", "end").strip()
        if not text:
            messagebox.showinfo("Nothing to translate", "Type something in the source box first.")
            return

        src_lang = LANGUAGES[self.source_lang_var.get()]
        tgt_lang = LANGUAGES[self.target_lang_var.get()]

        self.translate_btn.configure(state="disabled")
        self._set_status("Translating...", busy=True)

        def worker():
            try:
                result = self.translator.translate(text, src_lang, tgt_lang)
                error = None
            except Exception as exc:
                result = None
                error = exc
            self.root.after(0, self._on_translation_done, result, error, tgt_lang)

        threading.Thread(target=worker, daemon=True).start()

    def _on_translation_done(self, result, error, tgt_lang):
        self.translate_btn.configure(state="normal")

        if error is not None:
            self._set_status("Translation failed.", busy=False)
            messagebox.showerror("Translation Error", str(error))
            return

        self._set_output_font(tgt_lang)
        self.output_text.configure(state="normal")
        self.output_text.delete("1.0", "end")
        self.output_text.insert("1.0", result)
        self.output_text.configure(state="disabled")
        self._set_status("Ready.", busy=False)

    def _on_clear(self):
        self.input_text.delete("1.0", "end")
        self.output_text.configure(state="normal")
        self.output_text.delete("1.0", "end")
        self.output_text.configure(state="disabled")

    def _on_copy_output(self):
        content = self.output_text.get("1.0", "end").strip()
        if not content:
            return
        self.root.clipboard_clear()
        self.root.clipboard_append(content)
        self._set_status("Output copied to clipboard.", busy=False)

    def _on_swap_languages(self):
        src, tgt = self.source_lang_var.get(), self.target_lang_var.get()
        self.source_lang_var.set(tgt)
        self.target_lang_var.set(src)

        input_content = self.input_text.get("1.0", "end").strip()
        output_content = self.output_text.get("1.0", "end").strip()

        self.input_text.delete("1.0", "end")
        self.input_text.insert("1.0", output_content)

        self.output_text.configure(state="normal")
        self.output_text.delete("1.0", "end")
        self.output_text.insert("1.0", input_content)
        self.output_text.configure(state="disabled")

    def _on_target_lang_changed(self, *_args):
        tgt_lang = LANGUAGES.get(self.target_lang_var.get())
        if tgt_lang:
            self._set_output_font(tgt_lang)

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _set_output_font(self, tgt_lang_code):
        font_name = self.fonts.for_lang_code(tgt_lang_code)
        was_disabled = self.output_text.cget("state") == "disabled"
        if was_disabled:
            self.output_text.configure(state="normal")
        self.output_text.configure(font=(font_name, 13))
        if was_disabled:
            self.output_text.configure(state="disabled")

    def _set_status(self, message, busy):
        prefix = "\u23f3 " if busy else ""
        self.status_var.set(f"{prefix}{message}")


def main():
    root = tk.Tk()
    TranslatorGUI(root)
    root.mainloop()


if __name__ == "__main__":
    main()

Loading models on cuda...


## 5. Evaluation & Analysis

**Evaluation Method:**
- Qualitative evaluation of translation accuracy
- Performance observation (response time)

**Sample Output:**
- Hindi → English
- English → Tamil

**Limitations:**
- Grammar errors in complex sentences
- Large model size for very low-end systems

## 6. Ethical Considerations & Responsible AI

- Possible bias inherited from training data
- Unequal language representation
- No user data is stored or transmitted
- Offline inference ensures privacy
- Responsible usage disclaimer applied

## 7. Conclusion & Future Scope

**Conclusion:**
The Offline LTS successfully enables multilingual text translation without internet dependency while maintaining acceptable accuracy and performance.

**Future Scope:**
- Add more Indian and global languages
- Speech-to-text and text-to-speech integration
- Mobile and embedded deployment
- Model fine-tuning for domain-specific accuracy

In [4]:
print(repr(translator.translate("how are you", "english", "hindi")))
print(repr(translator.translate("how are you", "english", "punjabi")))
print(repr(translator.translate("good morning", "english", "hindi")))

'आप कैसे हैं?'
'ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ?'
'सुप्रभात।'
